<a href="https://colab.research.google.com/github/Manojmp7676/NATURAL-LANGUAGE-PROCESSING-NLP-/blob/main/NLP_FINAL_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok scikit-learn pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 2.8 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
"""
AI-Generated vs Human Content Detection
NLP Mini Project - Streamlit App (Single File, Colab-ready)
Author: Manoj MP

Model: TF-IDF (word + char n-grams) + Logistic Regression
Dataset: Embedded (no external download needed) - human vs AI-written text samples
"""

import streamlit as st
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import FeatureUnion
import plotly.graph_objects as go
import plotly.express as px

# ------------------------------------------------------------------
# 1. PAGE CONFIG + CUSTOM STYLING
# ------------------------------------------------------------------
st.set_page_config(
    page_title="AI vs Human Text Detector",
    page_icon="🕵️",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700;800;900&family=Inter:wght@400;500;600;700&display=swap');

    html, body, [class*="css"]  { font-family: 'Inter', sans-serif; }

    /* ---------- ANIMATED BACKGROUND ---------- */
    .stApp {
        background:
            radial-gradient(circle at 15% 15%, rgba(127,90,240,0.16), transparent 42%),
            radial-gradient(circle at 85% 10%, rgba(44,182,125,0.14), transparent 45%),
            radial-gradient(circle at 75% 85%, rgba(255,107,107,0.12), transparent 45%),
            radial-gradient(circle at 10% 80%, rgba(56,189,248,0.10), transparent 40%),
            linear-gradient(180deg, #0a0c14 0%, #0d0f18 40%, #090b11 100%);
        background-attachment: fixed;
    }
    .stApp::before {
        content: "";
        position: fixed;
        inset: 0;
        background-image:
            linear-gradient(rgba(255,255,255,0.025) 1px, transparent 1px),
            linear-gradient(90deg, rgba(255,255,255,0.025) 1px, transparent 1px);
        background-size: 44px 44px;
        pointer-events: none;
        z-index: 0;
    }
    .blob {
        position: fixed;
        border-radius: 50%;
        filter: blur(100px);
        z-index: 0;
        pointer-events: none;
        animation: floatBlob 14s ease-in-out infinite;
    }
    .blob1 { width: 420px; height: 420px; background: #7F5AF0; opacity: 0.30; top: -120px; left: -100px; animation-delay: 0s; }
    .blob2 { width: 380px; height: 380px; background: #2CB67D; opacity: 0.24; bottom: -140px; right: -80px; animation-delay: 3s; }
    .blob3 { width: 300px; height: 300px; background: #FF6B6B; opacity: 0.20; top: 45%; right: 8%; animation-delay: 6s; }
    .blob4 { width: 260px; height: 260px; background: #38BDF8; opacity: 0.18; top: 65%; left: 5%; animation-delay: 9s; }
    @keyframes floatBlob {
        0%, 100% { transform: translate(0,0) scale(1); }
        33% { transform: translate(40px, -35px) scale(1.08); }
        66% { transform: translate(-25px, 25px) scale(0.95); }
    }

    section.main .block-container { position: relative; z-index: 1; padding-top: 1.5rem; }

    /* ---------- HERO SECTION ---------- */
    .hero {
        background: linear-gradient(135deg, rgba(127,90,240,0.10), rgba(44,182,125,0.06) 60%, rgba(255,107,107,0.05));
        border: 1px solid rgba(127,90,240,0.28);
        border-radius: 26px;
        padding: 2.4rem 2.6rem;
        margin-bottom: 1.8rem;
        backdrop-filter: blur(14px);
        box-shadow: 0 0 50px rgba(127,90,240,0.10), inset 0 1px 0 rgba(255,255,255,0.04);
    }
    .big-title {
        font-family: 'Poppins', sans-serif;
        font-size: 3.1rem;
        font-weight: 800;
        line-height: 1.15;
        background: linear-gradient(90deg, #A78BFA, #2CB67D, #38BDF8, #A78BFA);
        background-size: 300% auto;
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        margin-bottom: 0.3rem;
        animation: shine 8s linear infinite;
        filter: drop-shadow(0 0 22px rgba(127,90,240,0.25));
    }
    @keyframes shine { to { background-position: 300% center; } }
    .subtitle {
        color: #B8B8D0;
        font-size: 1.12rem;
        margin-top: 0.1rem;
        margin-bottom: 1.4rem;
        max-width: 720px;
    }
    .badge-row { display: flex; flex-wrap: wrap; gap: 0.55rem; }
    .badge {
        display: inline-flex;
        align-items: center;
        background: linear-gradient(135deg, rgba(127,90,240,0.16), rgba(44,182,125,0.10));
        border: 1px solid rgba(167,139,250,0.35);
        color: #E4E0FF;
        padding: 0.35rem 0.95rem;
        border-radius: 20px;
        font-size: 0.8rem;
        font-weight: 500;
        box-shadow: 0 0 14px rgba(127,90,240,0.08);
    }

    /* ---------- RESULT CARD ---------- */
    .result-card {
        padding: 2.2rem;
        border-radius: 22px;
        text-align: center;
        margin-top: 1rem;
        margin-bottom: 1rem;
        backdrop-filter: blur(10px);
        position: relative;
        overflow: hidden;
    }
    .human-card {
        background: linear-gradient(145deg, rgba(44,182,125,0.20), rgba(44,182,125,0.04));
        border: 1.5px solid #2CB67D;
        box-shadow: 0 0 50px rgba(44,182,125,0.22), inset 0 1px 0 rgba(255,255,255,0.05);
    }
    .ai-card {
        background: linear-gradient(145deg, rgba(255,107,107,0.20), rgba(255,107,107,0.04));
        border: 1.5px solid #FF6B6B;
        box-shadow: 0 0 50px rgba(255,107,107,0.22), inset 0 1px 0 rgba(255,255,255,0.05);
    }
    .result-card h1 { font-family: 'Poppins', sans-serif; }

    /* ---------- CHART / GAUGE WRAPPER ---------- */
    .chart-card {
        background: linear-gradient(160deg, rgba(26,29,46,0.85), rgba(14,16,23,0.9));
        border: 1px solid #262a45;
        border-radius: 22px;
        padding: 0.6rem 0.8rem 0.2rem 0.8rem;
        box-shadow: 0 8px 30px rgba(0,0,0,0.3);
    }

    /* ---------- STAT BOXES ---------- */
    .stat-box {
        background: linear-gradient(160deg, #1B1E30, #12141F);
        border-radius: 16px;
        padding: 1.2rem 0.6rem;
        text-align: center;
        border: 1px solid #2A2E4A;
        transition: all 0.25s ease;
        position: relative;
    }
    .stat-box:hover {
        transform: translateY(-4px);
        border-color: #7F5AF0;
        box-shadow: 0 10px 30px rgba(127,90,240,0.25);
    }
    .stat-icon { font-size: 1.3rem; margin-bottom: 0.2rem; }
    .stat-number {
        font-size: 1.75rem;
        font-weight: 700;
        color: #A78BFA;
        font-family: 'Poppins', sans-serif;
        text-shadow: 0 0 18px rgba(167,139,250,0.35);
    }
    .stat-label {
        color: #9494B0;
        font-size: 0.8rem;
        margin-top: 0.25rem;
        letter-spacing: 0.02em;
    }

    /* ---------- GENERIC SECTION CARD ---------- */
    .section-card {
        background: linear-gradient(160deg, rgba(20,22,31,0.85), rgba(14,17,23,0.9));
        border: 1px solid #23253A;
        border-radius: 20px;
        padding: 1.6rem 1.8rem;
        margin-bottom: 1.2rem;
        box-shadow: 0 8px 30px rgba(0,0,0,0.25);
    }

    /* ---------- BUTTONS ---------- */
    .stButton > button {
        background: linear-gradient(135deg, #7F5AF0, #2CB67D) !important;
        color: white !important;
        border: none !important;
        border-radius: 14px !important;
        font-weight: 600 !important;
        padding: 0.7rem 1rem !important;
        box-shadow: 0 6px 24px rgba(127,90,240,0.35) !important;
        transition: all 0.25s ease !important;
        letter-spacing: 0.01em;
    }
    .stButton > button:hover {
        transform: translateY(-2px);
        box-shadow: 0 10px 32px rgba(127,90,240,0.5) !important;
        filter: brightness(1.08);
    }
    .stDownloadButton > button {
        background: linear-gradient(135deg, #1B1E30, #12141F) !important;
        color: #C9C9E8 !important;
        border: 1px solid #7F5AF0 !important;
        border-radius: 14px !important;
        font-weight: 600 !important;
        transition: all 0.25s ease !important;
    }
    .stDownloadButton > button:hover {
        border-color: #A78BFA !important;
        box-shadow: 0 6px 22px rgba(127,90,240,0.3) !important;
    }

    /* ---------- INPUTS ---------- */
    .stTextArea textarea, .stTextInput input {
        background-color: #12141F !important;
        border: 1px solid #2A2E4A !important;
        border-radius: 14px !important;
        color: #E4E4F0 !important;
    }
    .stTextArea textarea:focus, .stTextInput input:focus {
        border-color: #7F5AF0 !important;
        box-shadow: 0 0 0 2px rgba(127,90,240,0.25) !important;
    }
    div[data-baseweb="select"] > div {
        background-color: #12141F !important;
        border-color: #2A2E4A !important;
        border-radius: 12px !important;
    }

    /* ---------- METRICS ---------- */
    div[data-testid="stMetric"] {
        background: linear-gradient(160deg, #1B1E30, #12141F);
        border: 1px solid #2A2E4A;
        border-radius: 16px;
        padding: 0.8rem 1rem;
    }
    div[data-testid="stMetricValue"] { color: #A78BFA; font-family: 'Poppins', sans-serif; }

    /* ---------- TABS ---------- */
    .stTabs [data-baseweb="tab-list"] { gap: 8px; border-bottom: 1px solid #23253A; }
    .stTabs [data-baseweb="tab"] {
        background-color: #14161F;
        border: 1px solid #22243A;
        border-radius: 14px 14px 0 0;
        padding: 12px 22px;
        color: #9090A5;
        font-weight: 500;
        transition: all 0.2s ease;
    }
    .stTabs [data-baseweb="tab"]:hover { color: #C9C9E8; background-color: #1A1D2E; }
    .stTabs [aria-selected="true"] {
        background: linear-gradient(160deg, #1E2038, #171928) !important;
        color: #A78BFA !important;
        border-bottom: 2.5px solid #7F5AF0 !important;
        box-shadow: 0 -4px 20px rgba(127,90,240,0.15);
    }

    /* ---------- SIDEBAR ---------- */
    section[data-testid="stSidebar"] {
        background: linear-gradient(180deg, #0d0f18, #0a0c14);
        border-right: 1px solid #23253A;
    }

    /* ---------- EXPANDER ---------- */
    .streamlit-expanderHeader, div[data-testid="stExpander"] {
        background-color: #12141F !important;
        border: 1px solid #23253A !important;
        border-radius: 14px !important;
    }

    /* ---------- SCROLLBAR ---------- */
    ::-webkit-scrollbar { width: 10px; height: 10px; }
    ::-webkit-scrollbar-track { background: #0d0f18; }
    ::-webkit-scrollbar-thumb { background: linear-gradient(#7F5AF0, #2CB67D); border-radius: 10px; }

    /* ---------- FOOTER ---------- */
    .footer {
        text-align: center;
        color: #5A5A70;
        margin-top: 3rem;
        font-size: 0.85rem;
        padding-top: 1.5rem;
        border-top: 1px solid #23253A;
    }
</style>

<div class="blob blob1"></div>
<div class="blob blob2"></div>
<div class="blob blob3"></div>
<div class="blob blob4"></div>
""", unsafe_allow_html=True)

st.markdown("""
<div class="hero">
    <p class="big-title">🕵️ AI vs Human Text Detector</p>
    <p class="subtitle">An NLP model that predicts whether a piece of text was written by a human or generated by AI — complete with batch analysis, live confidence scoring, and full model insights.</p>
    <div class="badge-row">
        <span class="badge">🧠 TF-IDF + Logistic Regression</span>
        <span class="badge">📊 80-sample dataset</span>
        <span class="badge">⚡ Real-time inference</span>
        <span class="badge">📁 Batch mode</span>
        <span class="badge">🎯 Explainable predictions</span>
    </div>
</div>
""", unsafe_allow_html=True)

# ------------------------------------------------------------------
# 2. EMBEDDED DATASET (Human-written vs AI-generated text samples)
# ------------------------------------------------------------------
human_texts = [
    "honestly i dont even know why i stayed up till 3am watching random youtube videos again lol",
    "my mom called me twice today and i missed both calls, feeling super guilty rn ngl",
    "so i tried making pasta from scratch and it was a total disaster, sauce everywhere",
    "cant believe the auto guy charged me 200 for a ride that shouldve been 80, bengaluru traffic is wild",
    "just finished my internship report at like midnight, eyes are literally burning",
    "went to my cousin's wedding last weekend, the food was amazing but i danced way too much and my legs hurt",
    "ugh mondays are the worst, i overslept and missed the first lecture again",
    "found this old photo of me and my dog from like 5 years ago and now im crying a little",
    "tried to fix my laptop myself and ended up making it worse, calling the repair guy tomorrow",
    "my professor gave us a surprise quiz today and i genuinely blanked on question 3",
    "spent the whole evening arguing with my brother about whose turn it is to do dishes",
    "finally got my bike serviced, it feels so much smoother now, worth every rupee",
    "i swear the wifi in my hostel dies exactly when i have an important zoom call",
    "made maggi at 1am because i was too lazy to cook anything proper, no regrets though",
    "got stuck in bengaluru rain without an umbrella, reached home looking like i took a shower with clothes on",
    "my friend ghosted our group project and now im doing all the slides alone, so frustrated",
    "tried a new cafe near my college, the coffee was mid but the vibe was actually really nice",
    "i keep forgetting to water my plants and now one of them is basically dead lol oops",
    "binge watched an entire season in one night and now i regret literally everything about my life choices",
    "dad wants me to learn driving this month, im honestly kind of scared of the traffic here",
    "spent way too much money on books this month but honestly zero regrets, worth it",
    "my roommate snores so loud i had to sleep with earphones in again last night",
    "finally beat that boss in the game after like 15 tries, felt so satisfying not gonna lie",
    "went grocery shopping and completely forgot to buy the one thing i actually needed, milk",
    "the metro was so packed today i literally couldnt move my arms, bengaluru rush hour is no joke",
    "tried explaining my project to my grandma and she just nodded and asked if i was eating well",
    "i accidentally sent a text meant for my friend to my professor, wanted to disappear immediately",
    "power went out right when i was about to submit my assignment, panicked for a solid ten minutes",
    "my neighbor's kid keeps ringing our doorbell and running away, kind of annoying but also funny",
    "finally cleaned my room after like 3 weeks, found my missing charger under the bed of all places",
    "tried to cook biryani for the first time, it turned out okay but way too spicy, still ate it all",
    "watched the sunset from my terrace today, small things like that just make the day feel better",
    "got scolded by my manager for a silly excel mistake, felt bad the whole afternoon honestly",
    "my phone battery died during an important call, had to run around the house looking for a charger",
    "tried learning guitar from youtube, my fingers hurt so bad after just ten minutes of practice",
    "the auto driver took a shortcut i didnt even know existed, saved me like 20 minutes today",
    "spent my entire sunday just sleeping and eating, zero productivity but honestly needed it",
    "my little sister broke my earphones again, this is like the third pair this year",
    "finally submitted my resume after editing it for the hundredth time, hoping for the best now",
    "college wifi crashed right during the online exam, had to restart everything and lost ten minutes",
]

ai_texts = [
    "The rapid advancement of artificial intelligence has significantly transformed various industries, enabling enhanced efficiency and innovative solutions across multiple sectors.",
    "In conclusion, it is evident that effective time management plays a crucial role in achieving both personal and professional success in today's fast-paced world.",
    "This report aims to provide a comprehensive overview of the current market trends, highlighting key opportunities and challenges faced by organizations globally.",
    "It is important to note that regular physical exercise contributes significantly to overall well-being, improving both mental clarity and physical health outcomes.",
    "The implementation of renewable energy sources has proven to be a sustainable solution for reducing carbon emissions and mitigating the effects of climate change.",
    "Furthermore, effective communication skills are essential for building strong professional relationships and fostering collaborative work environments within organizations.",
    "The following analysis examines the impact of digital transformation on modern business operations, focusing on efficiency, scalability, and customer engagement.",
    "Overall, the data suggests a consistent upward trend in consumer preferences toward environmentally sustainable products and services over the past decade.",
    "This document outlines the key strategies necessary for optimizing organizational performance while maintaining a strong focus on employee satisfaction and retention.",
    "As a result, businesses must adapt to evolving technological landscapes in order to remain competitive within an increasingly globalized economic environment.",
    "The purpose of this study is to explore the correlation between educational attainment and long-term career growth across various professional domains.",
    "In summary, the findings indicate that a balanced diet combined with consistent exercise significantly reduces the risk of chronic health conditions.",
    "Effective leadership requires a combination of strategic vision, clear communication, and the ability to inspire and motivate team members toward common goals.",
    "The integration of machine learning algorithms into everyday applications has substantially improved the accuracy and efficiency of predictive analytics systems.",
    "It is essential to consider multiple perspectives when addressing complex societal issues, as this approach fosters more inclusive and effective solutions.",
    "This analysis highlights the importance of data-driven decision making in enhancing organizational efficiency and achieving measurable business outcomes.",
    "The advent of cloud computing has revolutionized data storage and accessibility, allowing organizations to scale their operations with greater flexibility.",
    "Consequently, the adoption of sustainable practices has become a fundamental priority for organizations seeking to minimize their environmental footprint.",
    "A comprehensive understanding of consumer behavior is critical for developing effective marketing strategies that resonate with target demographics.",
    "The research findings demonstrate a significant correlation between employee engagement and overall organizational productivity across diverse industries.",
    "In light of recent technological developments, it is imperative that educational institutions adapt their curricula to meet evolving industry demands.",
    "This section provides an in-depth examination of the various factors influencing customer satisfaction within the competitive retail sector.",
    "The utilization of automation technologies has streamlined operational processes, resulting in significant improvements in both efficiency and cost reduction.",
    "Moreover, fostering a culture of continuous learning within organizations enables employees to adapt more effectively to changing market conditions.",
    "The study concludes that proactive risk management strategies are essential for ensuring long-term organizational resilience and sustainability.",
    "It can be observed that global supply chains have become increasingly interconnected, necessitating robust contingency planning frameworks.",
    "The following recommendations are proposed to enhance operational efficiency while simultaneously reducing overall resource consumption and waste.",
    "Artificial intelligence systems are increasingly being deployed to automate routine tasks, thereby allowing human resources to focus on strategic initiatives.",
    "This comprehensive framework outlines the essential components required for successful project management within complex organizational structures.",
    "The analysis reveals that consistent innovation is a key determinant of long-term competitive advantage in rapidly evolving market landscapes.",
    "In order to achieve sustainable growth, organizations must prioritize strategic planning alongside effective resource allocation and management practices.",
    "The evidence suggests that early intervention strategies significantly improve outcomes across a wide range of educational and developmental contexts.",
    "This report emphasizes the critical importance of cybersecurity measures in safeguarding sensitive organizational data against emerging digital threats.",
    "Ultimately, the successful implementation of these strategies requires strong leadership commitment and cross-functional collaboration throughout the organization.",
    "The findings underscore the necessity of adopting a holistic approach when addressing multifaceted challenges within organizational ecosystems.",
    "It is widely acknowledged that effective stakeholder engagement is fundamental to the successful execution of large-scale organizational initiatives.",
    "This paper explores the implications of emerging technologies on workforce dynamics and the future landscape of professional employment opportunities.",
    "The results clearly indicate that strategic investment in employee development yields substantial long-term returns for organizational performance.",
    "Given the current economic climate, organizations must remain agile and responsive to rapidly changing market conditions and consumer expectations.",
    "In conclusion, the comprehensive analysis presented herein provides valuable insights for stakeholders seeking to navigate an increasingly complex business environment.",
]

texts = human_texts + ai_texts
labels = ["Human"] * len(human_texts) + ["AI"] * len(ai_texts)
df = pd.DataFrame({"text": texts, "label": labels})


# ------------------------------------------------------------------
# 3. TEXT FEATURE HELPERS (used for both model + UI stats)
# ------------------------------------------------------------------
def clean_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def text_stats(text):
    words = text.split()
    sentences = re.split(r'[.!?]+', text)
    sentences = [s for s in sentences if s.strip()]
    word_count = len(words)
    sentence_count = max(len(sentences), 1)
    avg_word_len = np.mean([len(w) for w in words]) if words else 0
    unique_words = len(set(w.lower() for w in words))
    lexical_diversity = unique_words / word_count if word_count > 0 else 0
    avg_sentence_len = word_count / sentence_count
    return {
        "Word Count": word_count,
        "Sentence Count": sentence_count,
        "Avg Word Length": round(avg_word_len, 2),
        "Lexical Diversity": round(lexical_diversity, 2),
        "Avg Sentence Length": round(avg_sentence_len, 2),
    }


# ------------------------------------------------------------------
# 4. TRAIN MODEL (cached so it trains only once per session)
# ------------------------------------------------------------------
@st.cache_resource
def train_model():
    df_clean = df.copy()
    df_clean["clean_text"] = df_clean["text"].apply(clean_text)

    X_train, X_test, y_train, y_test = train_test_split(
        df_clean["clean_text"], df_clean["label"],
        test_size=0.2, random_state=42, stratify=df_clean["label"]
    )

    # Word-level + character-level n-grams capture both vocabulary
    # choice (AI text uses more formal words) and writing style/typos.
    word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=3000, sublinear_tf=True)
    char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=3000)

    vectorizer = FeatureUnion([("word", word_vec), ("char", char_vec)])

    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model = LogisticRegression(max_iter=1000, C=2.0)
    model.fit(X_train_vec, y_train)

    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)

    return vectorizer, model, acc, report


with st.spinner("Training model on startup... (only happens once)"):
    vectorizer, model, test_accuracy, report = train_model()


# ------------------------------------------------------------------
# 5. SESSION STATE (prediction history)
# ------------------------------------------------------------------
if "history" not in st.session_state:
    st.session_state.history = []

def predict_text(text):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    proba = model.predict_proba(vec)[0]
    prob_dict = dict(zip(model.classes_, proba))
    confidence = prob_dict[pred] * 100
    return pred, confidence, prob_dict, cleaned


def make_gauge(confidence, pred):
    color = "#2CB67D" if pred == "Human" else "#FF6B6B"
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=confidence,
        number={"suffix": "%", "font": {"size": 34, "color": color}},
        gauge={
            "axis": {"range": [0, 100], "tickcolor": "#555"},
            "bar": {"color": color},
            "bgcolor": "#1A1D2E",
            "borderwidth": 0,
            "steps": [
                {"range": [0, 50], "color": "#20222E"},
                {"range": [50, 100], "color": "#262A3A"},
            ],
        },
        title={"text": "Confidence", "font": {"size": 14, "color": "#9090A5"}}
    ))
    fig.update_layout(
        height=220, margin=dict(l=20, r=20, t=40, b=10),
        paper_bgcolor="rgba(0,0,0,0)", font={"color": "#e0e0e0"}
    )
    return fig


# ------------------------------------------------------------------
# 6. SIDEBAR - PROJECT INFO
# ------------------------------------------------------------------
with st.sidebar:
    st.header("ℹ️ About this Project")
    st.markdown("""
    **Goal:** Classify text as *Human-written* or *AI-generated*.

    **Pipeline:**
    1. Clean & normalize text
    2. TF-IDF (word 1-2 grams + char 3-5 grams)
    3. Logistic Regression classifier

    **Dataset:** 80 labeled samples
    (40 Human / 40 AI-style text), embedded directly
    in this script.
    """)
    st.divider()
    st.metric("Model Test Accuracy", f"{test_accuracy*100:.1f}%")
    st.caption(f"Dataset size: {len(df)} samples")

    st.divider()
    st.subheader("🕘 Recent Predictions")
    if st.session_state.history:
        for h in reversed(st.session_state.history[-5:]):
            icon = "🧑" if h["Prediction"] == "Human" else "🤖"
            st.caption(f"{icon} {h['Prediction']} ({h['Confidence']}%) — {h['Text'][:35]}...")
        if st.button("🗑️ Clear History", use_container_width=True):
            st.session_state.history = []
            st.rerun()
    else:
        st.caption("No predictions yet. Analyze some text to see history here.")


# ------------------------------------------------------------------
# 7. MAIN TABS
# ------------------------------------------------------------------
tab1, tab2, tab3, tab4 = st.tabs(["🔍 Analyze Text", "📁 Batch Analysis", "📊 Dataset & Model Insights", "🕘 History"])

# =====================  TAB 1: SINGLE TEXT ANALYSIS  =====================
with tab1:
    st.subheader("Paste text below to analyze")

    example_choice = st.selectbox(
        "Or try a quick example:",
        ["-- Select an example --", "Human example", "AI example"]
    )

    default_text = ""
    if example_choice == "Human example":
        default_text = "honestly i have no idea what im doing with my life rn but i just went for a walk and felt a bit better lol"
    elif example_choice == "AI example":
        default_text = "The integration of advanced technologies has significantly enhanced operational efficiency across a wide range of industries, resulting in measurable improvements in productivity."

    user_text = st.text_area(
        "Enter text (at least a sentence or two for best results):",
        value=default_text, height=150, placeholder="Paste any paragraph here..."
    )

    analyze_btn = st.button("🔍 Analyze Text", type="primary", use_container_width=True)

    if analyze_btn:
        if not user_text.strip() or len(user_text.split()) < 3:
            st.warning("Please enter a longer piece of text (at least a few words) for a meaningful prediction.")
        else:
            pred, confidence, prob_dict, cleaned = predict_text(user_text)

            st.session_state.history.append({
                "Time": datetime.now().strftime("%H:%M:%S"),
                "Text": user_text.strip(),
                "Prediction": pred,
                "Confidence": round(confidence, 1)
            })

            card_class = "human-card" if pred == "Human" else "ai-card"
            emoji = "🧑" if pred == "Human" else "🤖"

            col_left, col_right = st.columns([1.3, 1])
            with col_left:
                st.markdown(f"""
                <div class="result-card {card_class}">
                    <h1 style="margin-bottom:0;">{emoji} {pred}-Written</h1>
                    <p style="font-size:1.1rem; margin-top:0.3rem;">Confidence: <b>{confidence:.1f}%</b></p>
                </div>
                """, unsafe_allow_html=True)

                st.markdown("**Prediction Breakdown**")
                c1, c2 = st.columns(2)
                with c1:
                    st.markdown("🧑 Human")
                    st.progress(float(prob_dict.get("Human", 0)))
                    st.caption(f"{prob_dict.get('Human', 0)*100:.1f}%")
                with c2:
                    st.markdown("🤖 AI")
                    st.progress(float(prob_dict.get("AI", 0)))
                    st.caption(f"{prob_dict.get('AI', 0)*100:.1f}%")

            with col_right:
                st.markdown('<div class="chart-card">', unsafe_allow_html=True)
                st.plotly_chart(make_gauge(confidence, pred), use_container_width=True)
                st.markdown('</div>', unsafe_allow_html=True)

            st.markdown("**Text Statistics**")
            stat_icons = {
                "Word Count": "📝", "Sentence Count": "✂️", "Avg Word Length": "🔤",
                "Lexical Diversity": "🎯", "Avg Sentence Length": "📏"
            }
            stats = text_stats(user_text)
            cols = st.columns(len(stats))
            for col, (label, value) in zip(cols, stats.items()):
                with col:
                    st.markdown(f"""
                    <div class="stat-box">
                        <div class="stat-icon">{stat_icons.get(label, "📊")}</div>
                        <div class="stat-number">{value}</div>
                        <div class="stat-label">{label}</div>
                    </div>
                    """, unsafe_allow_html=True)

            with st.expander("🔬 See what influenced this prediction"):
                try:
                    word_vectorizer = vectorizer.transformer_list[0][1]
                    word_features = word_vectorizer.get_feature_names_out()
                    word_vec_only = word_vectorizer.transform([cleaned])
                    coefs = model.coef_[0][:len(word_features)]
                    nonzero_idx = word_vec_only.nonzero()[1]
                    contributions = [(word_features[i], coefs[i] * word_vec_only[0, i]) for i in nonzero_idx]
                    contributions.sort(key=lambda x: abs(x[1]), reverse=True)
                    top_contrib = contributions[:10]

                    if top_contrib:
                        contrib_df = pd.DataFrame(top_contrib, columns=["Word/Phrase", "Influence Score"])
                        contrib_df["Leans Toward"] = contrib_df["Influence Score"].apply(lambda x: "AI" if x > 0 else "Human")
                        fig = px.bar(
                            contrib_df.sort_values("Influence Score"), x="Influence Score", y="Word/Phrase",
                            color="Leans Toward", orientation="h",
                            color_discrete_map={"AI": "#FF6B6B", "Human": "#2CB67D"}
                        )
                        fig.update_layout(
                            height=350, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                            font={"color": "#e0e0e0"}, margin=dict(l=10, r=10, t=20, b=10)
                        )
                        st.plotly_chart(fig, use_container_width=True)
                    else:
                        st.caption("No strong individual word signals found for this text.")
                except Exception:
                    st.caption("Feature breakdown unavailable for this input.")

# =====================  TAB 2: BATCH ANALYSIS  =====================
with tab2:
    st.subheader("Analyze multiple texts at once")
    st.caption("Paste one text per line below, or upload a CSV with a column named 'text'.")

    batch_input = st.text_area(
        "One text sample per line:",
        height=180,
        placeholder="i just walked my dog and its raining, kinda nice actually\nThe implementation of this framework significantly enhances operational efficiency.\n..."
    )

    uploaded_file = st.file_uploader("Or upload a CSV file (must contain a 'text' column)", type=["csv"])

    run_batch = st.button("📁 Run Batch Analysis", type="primary", use_container_width=True)

    if run_batch:
        batch_texts = []
        if uploaded_file is not None:
            try:
                up_df = pd.read_csv(uploaded_file)
                if "text" in up_df.columns:
                    batch_texts = up_df["text"].dropna().astype(str).tolist()
                else:
                    st.error("CSV must contain a column named 'text'.")
            except Exception as e:
                st.error(f"Could not read CSV: {e}")
        elif batch_input.strip():
            batch_texts = [line.strip() for line in batch_input.split("\n") if line.strip()]

        if not batch_texts:
            st.warning("Please paste some text lines or upload a valid CSV first.")
        else:
            results = []
            progress = st.progress(0)
            for i, t in enumerate(batch_texts):
                if len(t.split()) >= 3:
                    pred, conf, _, _ = predict_text(t)
                    results.append({"Text": t, "Prediction": pred, "Confidence (%)": round(conf, 1)})
                progress.progress((i + 1) / len(batch_texts))

            result_df = pd.DataFrame(results)
            st.success(f"Analyzed {len(result_df)} texts.")

            colA, colB = st.columns([2, 1])
            with colA:
                def highlight_pred(row):
                    color = "#2CB67D22" if row["Prediction"] == "Human" else "#FF6B6B22"
                    return [f"background-color: {color}"] * len(row)
                st.dataframe(result_df.style.apply(highlight_pred, axis=1), use_container_width=True, height=350)
            with colB:
                counts = result_df["Prediction"].value_counts().reset_index()
                counts.columns = ["Prediction", "Count"]
                fig = px.pie(
                    counts, names="Prediction", values="Count", hole=0.55,
                    color="Prediction", color_discrete_map={"Human": "#2CB67D", "AI": "#FF6B6B"}
                )
                fig.update_layout(
                    height=300, paper_bgcolor="rgba(0,0,0,0)", font={"color": "#e0e0e0"},
                    margin=dict(l=10, r=10, t=10, b=10), showlegend=True
                )
                st.plotly_chart(fig, use_container_width=True)

            csv_out = result_df.to_csv(index=False).encode("utf-8")
            st.download_button("⬇️ Download Results as CSV", csv_out, "batch_predictions.csv", "text/csv", use_container_width=True)

# =====================  TAB 3: DATASET & MODEL INSIGHTS  =====================
with tab3:
    st.subheader("Dataset Overview")

    col1, col2, col3 = st.columns(3)
    with col1:
        st.markdown(f"""<div class="stat-box"><div class="stat-icon">📚</div><div class="stat-number">{len(df)}</div><div class="stat-label">Total Samples</div></div>""", unsafe_allow_html=True)
    with col2:
        st.markdown(f"""<div class="stat-box"><div class="stat-icon">🧑</div><div class="stat-number">{len(human_texts)}</div><div class="stat-label">Human Samples</div></div>""", unsafe_allow_html=True)
    with col3:
        st.markdown(f"""<div class="stat-box"><div class="stat-icon">🤖</div><div class="stat-number">{len(ai_texts)}</div><div class="stat-label">AI Samples</div></div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    c1, c2 = st.columns(2)

    with c1:
        st.markdown("**Class Distribution**")
        dist = df["label"].value_counts().reset_index()
        dist.columns = ["Label", "Count"]
        fig = px.pie(dist, names="Label", values="Count", hole=0.55,
                     color="Label", color_discrete_map={"Human": "#2CB67D", "AI": "#FF6B6B"})
        fig.update_layout(height=300, paper_bgcolor="rgba(0,0,0,0)", font={"color": "#e0e0e0"}, margin=dict(l=10, r=10, t=10, b=10))
        st.plotly_chart(fig, use_container_width=True)

    with c2:
        st.markdown("**Avg. Word Count by Class**")
        df["word_count"] = df["text"].apply(lambda t: len(t.split()))
        avg_wc = df.groupby("label")["word_count"].mean().reset_index()
        fig = px.bar(avg_wc, x="label", y="word_count", color="label",
                     color_discrete_map={"Human": "#2CB67D", "AI": "#FF6B6B"})
        fig.update_layout(height=300, paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                           font={"color": "#e0e0e0"}, margin=dict(l=10, r=10, t=10, b=10), showlegend=False)
        st.plotly_chart(fig, use_container_width=True)

    st.divider()
    st.subheader("Model Performance")

    col1, col2 = st.columns([1, 1])
    with col1:
        st.metric("Test Accuracy", f"{test_accuracy*100:.1f}%")
        st.markdown("**Classification Report**")
        st.dataframe(pd.DataFrame(report).transpose().round(2), use_container_width=True)

    with col2:
        # Recompute confusion matrix on a fresh split for display
        df_clean = df.copy()
        df_clean["clean_text"] = df_clean["text"].apply(clean_text)
        X_train, X_test, y_train, y_test = train_test_split(
            df_clean["clean_text"], df_clean["label"], test_size=0.2, random_state=42, stratify=df_clean["label"]
        )
        X_test_vec = vectorizer.transform(X_test)
        y_pred = model.predict(X_test_vec)
        cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
        fig = px.imshow(cm, text_auto=True, x=model.classes_, y=model.classes_,
                         color_continuous_scale=["#14161F", "#7F5AF0"], labels=dict(x="Predicted", y="Actual"))
        fig.update_layout(height=300, paper_bgcolor="rgba(0,0,0,0)", font={"color": "#e0e0e0"}, margin=dict(l=10, r=10, t=30, b=10))
        st.markdown("**Confusion Matrix**")
        st.plotly_chart(fig, use_container_width=True)

    with st.expander("📁 View Full Dataset"):
        st.dataframe(df[["text", "label"]], use_container_width=True, height=300)

# =====================  TAB 4: HISTORY  =====================
with tab4:
    st.subheader("Your Prediction History")
    if st.session_state.history:
        hist_df = pd.DataFrame(st.session_state.history)
        st.dataframe(hist_df, use_container_width=True, height=350)

        csv_hist = hist_df.to_csv(index=False).encode("utf-8")
        st.download_button("⬇️ Download History as CSV", csv_hist, "prediction_history.csv", "text/csv", use_container_width=True)

        if st.button("🗑️ Clear All History", use_container_width=True):
            st.session_state.history = []
            st.rerun()
    else:
        st.info("No predictions yet. Head to the 'Analyze Text' or 'Batch Analysis' tab to get started.")

st.markdown('<p class="footer">Built with Streamlit + Plotly • TF-IDF + Logistic Regression • NLP Mini Project by Manoj MP</p>', unsafe_allow_html=True)

Writing app.py


In [ ]:
!pkill -f streamlit

In [ ]:
from google.colab import userdata

In [ ]:
!ls -la app.py

-rw-r--r-- 1 root root 40855 Aug 19 02:56 app.py


In [ ]:
!pkill -f streamlit
!pkill -f ngrok
import time
time.sleep(2)

In [ ]:
import subprocess
import time

process = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.address", "0.0.0.0"],
    stdout=open("/content/log.txt", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(10)
print("Process running:", process.poll() is None)

Process running: True


In [ ]:
!cat /content/log.txt



2026-08-19 02:57:27.922 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.24.252.133:8501



In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3Hs05JOXppShXYQ6LaBY13QAyt6_34dXXH9YNV9TbJ5cQCGwv")
public_url = ngrok.connect(8501)
print("Your app is live here:", public_url)

Your app is live here: NgrokTunnel: "https://recycler-rare-trickily.ngrok-free.dev" -> "http://localhost:8501"
